# D11b — quattro regole di riparto alternative, sul banco di D11

**Prima di eseguire qualunque cosa oltre la cella 3:** la pre-registrazione dentro
`PROMPT_D11b_allocatore_regole_alternative.md` deve essere approvata da Giacomo. La cella 4 e'
bloccata da `APPROVATO = False` finche' non la si cambia a mano.

**Runtime:** CPU basta. Niente Meridian, niente refit. Tutto lo studio dura pochi minuti.

**Cosa fa.** Riusa `studio_D11.py` (non modificato) e aggiunge `studio_D11b.py` con le regole
B1 (aggregato a 4), B2 (solo tagli), H9 (passo scalato con la certezza) e la misura H6 (il
totale). Stessi 8 casi, stessa verita', stesso nullo, stesso oracolo.

**Cosa serve su Drive**, in una cartella `BASE` (impostala nella cella 1):
- `D11_pacchetto/` — il pacchetto del 17 settembre (draw, canali, `proposta_budget.py`)
- `g5/genera_dati_simulati.py` — generatore ricostruito con G5, sha256[:16] = `c94aaee9d211d049`
- `D11_risultati.csv` — le attese del cancello

**Ordine:** cella 1 (controlli) → cella 2 (scrive gli script) → cella 3 (**cancello**: riproduce
D11, 8 su 8) → *approvazione* → cella 4 (studio) → cella 5 (lettura).


In [ ]:
# ============================================================================
# CELLA 1 - DOVE SONO I FILE, E CHE SIANO QUELLI GIUSTI
# ============================================================================
import hashlib, os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/MyDrive/MMM_D11b'      # <-- cambia se serve
except ImportError:
    BASE = os.path.abspath('.')                     # locale: la cartella corrente
os.chdir(BASE)
print('BASE =', BASE)

ATTESO_G5 = 'c94aaee9d211d049'
problemi = []
for p in ('D11_pacchetto/proposta_budget.py', 'D11_pacchetto/griglia/griglia_D1_geo_canali.csv',
          'D11_pacchetto/griglia_D10/griglia_D10_calendario_4giri_canali.csv',
          'g5/genera_dati_simulati.py', 'D11_risultati.csv'):
    if not os.path.exists(p):
        problemi.append('manca ' + p)
if os.path.exists('g5/genera_dati_simulati.py'):
    h = hashlib.sha256(open('g5/genera_dati_simulati.py', 'rb').read()).hexdigest()[:16]
    if h != ATTESO_G5:
        problemi.append('generatore con impronta ' + h + ' invece di ' + ATTESO_G5)
    else:
        print('generatore: impronta', h, 'OK')
import numpy; print('numpy', numpy.__version__)
if problemi:
    raise SystemExit(chr(10).join(problemi))
print('cella 1: tutto a posto')


In [ ]:
# ============================================================================
# CELLA 2 - SCRIVE studio_D11.py (di D11, invariato) E studio_D11b.py
# ============================================================================
import io, hashlib
S11 = '"""D11 - il riparto controfattuale: la regola 90/10 fa guadagnare?\n\nEsegue quanto pre-registrato in PREREGISTRAZIONE_D11_riparto_controfattuale.md (v2).\nPer ognuno degli 8 casi genera il mondo BASE con il generatore ricostruito + G5 e\nvaluta cinque ripartizioni sulla superficie di risposta vera, a `beta` FISSO:\n\n  (a) spesa attuale   (b) regola 90/10   (c) allocatore vecchio\n  (d) 200 riparti casuali della stessa cifra   (e) oracolo sulla stessa classe di mosse\n\nMetrica: contributo incrementale vero dei media, in candidature.\nuso: python studio_D11.py\n"""\nimport collections\nimport csv\nimport importlib.util\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nQUI = os.path.dirname(os.path.abspath(__file__))\nPK = os.path.join(QUI, "D11_pacchetto")\n_s = importlib.util.spec_from_file_location("g5", os.path.join(QUI, "g5", "genera_dati_simulati.py"))\nG = importlib.util.module_from_spec(_s); _s.loader.exec_module(G)\n_p = importlib.util.spec_from_file_location("pb", os.path.join(PK, "proposta_budget.py"))\nPB = importlib.util.module_from_spec(_p); _p.loader.exec_module(PB)\n\nCASI = [\n    ("D1", 101, "griglia/griglia_D1_geo_draw/D1_geo_m0101_draw.csv.gz", "griglia/griglia_D1_geo_canali.csv"),\n    ("D1", 103, "griglia/griglia_D1_geo_draw/D1_geo_m0103_draw.csv.gz", "griglia/griglia_D1_geo_canali.csv"),\n    ("D1", 105, "griglia/griglia_D1_geo_draw/D1_geo_m0105_draw.csv.gz", "griglia/griglia_D1_geo_canali.csv"),\n    ("D1", 107, "griglia/griglia_D1_geo_draw/D1_geo_m0107_draw.csv.gz", "griglia/griglia_D1_geo_canali.csv"),\n    ("D2", 102, "griglia/griglia_D2_calendario_draw/D2_cal_rot2_m0102_draw.csv.gz", "griglia/griglia_D2_calendario_canali.csv"),\n    ("D2", 103, "griglia/griglia_D2_calendario_draw/D2_cal_rot3_m0103_draw.csv.gz", "griglia/griglia_D2_calendario_canali.csv"),\n    ("D10", 42, "griglia_D10/griglia_D10_calendario_4giri_draw/D10_cal4_rot0_m0042_draw.csv.gz", "griglia_D10/griglia_D10_calendario_4giri_canali.csv"),\n    ("D10", 103, "griglia_D10/griglia_D10_calendario_4giri_draw/D10_cal4_rot3_m0103_draw.csv.gz", "griglia_D10/griglia_D10_calendario_4giri_canali.csv"),\n]\nCASO_GIA_VISTO = ("D10", 42)          # dichiarato nella pre-registrazione, §5\nN_CASUALI = 200\nSEME_NULLO = 20260917\nALLOC_VECCHIO_SU = "Google Ads"\nALLOC_VECCHIO_GIU = ["Subito Lavoro", "Altre job board", "Jooble"]\nALLOC_VECCHIO_LIMITE = 30.0\n\n\ndef mondo(seed):\n    """Il mondo BASE del seme: la superficie di risposta vera, pronta da valutare."""\n    p = G.genera(seed=seed)\n    canali = list(p["canali"])\n    pop = p["pop"]\n    curve = {}\n    for ch in canali:\n        c = G.CANALI[ch]\n        a = G.adstock_geometrico(p["ch_impr"][ch] / pop[None, :], c["lam"])\n        ec = c["ec_mult"] * float(np.mean(p["ch_impr"][ch].sum(axis=1)))\n        # K(g) = beta x pop_g x mg_g: il fattore che resta FISSO quando si sposta budget\n        K = p["beta"][ch] * (pop * p["beta_geo"][ch])\n        curve[ch] = (a, ec, c["slope"], K)\n    spesa = {ch: float(p["ch_spesa"][ch].sum()) for ch in canali}\n    return canali, spesa, curve\n\n\ndef contributo(curve, canali, riparto=None):\n    """Candidature incrementali vere totali sotto una ripartizione. beta FISSO."""\n    tot = 0.0\n    for ch in canali:\n        a, ec, slope, K = curve[ch]\n        k = 1.0 + (riparto or {}).get(ch, 0.0) / 100.0\n        tot += float((K[None, :] * G.hill(k * a, ec, slope)).sum())\n    return tot\n\n\ndef riparto_90_10(draw, can, seed):\n    spesa = {x["canale"]: float(x["spesa"]) for x in\n             csv.DictReader(open(os.path.join(PK, can), newline="", encoding="utf-8"))\n             if x["seed_mondo"] == str(seed)}\n    P, n = PB.probabilita(os.path.join(PK, draw), spesa)\n    righe, scambio = PB.proposta(P, spesa)\n    return ({c: d for c, p, dec, d, nu, no in righe if dec in ("ALZA", "TAGLIA")}, scambio, spesa)\n\n\ndef riparto_allocatore_vecchio(spesa):\n    """§6.2: Google al massimo (+30%), i tre job board al minimo (-30%). Il +30% di\n    Google chiede piu\' di quanto i tagli liberino, quindi l\'alzata si scala a cio\'\n    che i tagli liberano: stessa regola di bilancio di (b), spesa totale costante."""\n    liberati = sum(spesa[c] * ALLOC_VECCHIO_LIMITE / 100.0 for c in ALLOC_VECCHIO_GIU)\n    su_pieno = spesa[ALLOC_VECCHIO_SU] * ALLOC_VECCHIO_LIMITE / 100.0\n    scambio = min(su_pieno, liberati)\n    r = {c: -ALLOC_VECCHIO_LIMITE * (scambio / liberati) for c in ALLOC_VECCHIO_GIU}\n    r[ALLOC_VECCHIO_SU] = ALLOC_VECCHIO_LIMITE * (scambio / su_pieno)\n    return r, scambio\n\n\ndef coppie(canali):\n    return [(i, j) for i in canali for j in canali if i != j]\n\n\ndef main():\n    righe_out, dettaglio_nullo = [], {}\n    print("D11 - riparto controfattuale. Contributo vero dei media, in candidature.")\n    print("beta FISSO al mondo non modificato: si valuta la superficie di risposta vera.")\n    print()\n    for dis, seed, draw, can in CASI:\n        canali, spesa, curve = mondo(seed)\n        rip_b, eur_b, spesa_fit = riparto_90_10(draw, can, seed)\n        scarto_b = sum(spesa[c] * d / 100.0 for c, d in rip_b.items())\n        eur_b_mondo = sum(abs(spesa[c] * d / 100.0) for c, d in rip_b.items()) / 2.0\n        Ya = contributo(curve, canali)\n        Yb = contributo(curve, canali, rip_b)\n        rip_c, eur_c_fit = riparto_allocatore_vecchio(spesa)\n        Yc = contributo(curve, canali, rip_c)\n        eur_c = sum(abs(spesa[c] * d / 100.0) for c, d in rip_c.items()) / 2.0\n        # (d) controllo nullo: la stessa cifra fra due canali estratti a sorte\n        rng = np.random.default_rng(SEME_NULLO + seed)\n        tutte = coppie(canali)\n        gd = []\n        while len(gd) < N_CASUALI:\n            su, giu = tutte[int(rng.integers(0, len(tutte)))]\n            d_giu = -100.0 * eur_b_mondo / spesa[giu]\n            if d_giu <= -100.0:\n                continue\n            gd.append(contributo(curve, canali, {su: 100.0 * eur_b_mondo / spesa[su], giu: d_giu}) - Ya)\n        gd = np.array(sorted(gd))\n        # (e) oracolo: la migliore della stessa classe, stessa cifra\n        migliore, ge = None, -np.inf\n        for su, giu in tutte:\n            d_giu = -100.0 * eur_b_mondo / spesa[giu]\n            if d_giu <= -100.0:\n                continue\n            g = contributo(curve, canali, {su: 100.0 * eur_b_mondo / spesa[su], giu: d_giu}) - Ya\n            if g > ge:\n                ge, migliore = g, (su, giu)\n        gb, gc = Yb - Ya, Yc - Ya\n        p_b = float((gd < gb).mean())\n        f_b = gb / ge if ge > 0 else float("nan")\n        righe_out.append(dict(\n            disegno=dis, mondo=seed, gia_visto=int((dis, seed) == CASO_GIA_VISTO),\n            riparto_b=";".join(c + "=" + format(d, "+.6f") for c, d in rip_b.items()),\n            euro_mossi_b=round(eur_b_mondo, 2), scarto_somma_b_eur=round(scarto_b, 4),\n            Y_a=round(Ya, 3), Y_b=round(Yb, 3), Y_c=round(Yc, 3),\n            g_b=round(gb, 3), g_b_pct=round(100 * gb / Ya, 5),\n            g_c=round(gc, 3), g_c_pct=round(100 * gc / Ya, 5),\n            g_b_per_euro=round(gb * 40.0 / eur_b_mondo, 5),\n            g_c_per_euro=round(gc * 40.0 / eur_c, 5), euro_mossi_c=round(eur_c, 2),\n            g_d_mediana=round(float(np.median(gd)), 3),\n            g_d_p90=round(float(np.quantile(gd, 0.90)), 3),\n            g_d_min=round(float(gd.min()), 3), g_d_max=round(float(gd.max()), 3),\n            g_d_frazione_positive=round(float((gd > 0).mean()), 4),\n            g_e=round(ge, 3), g_e_pct=round(100 * ge / Ya, 5),\n            oracolo=migliore[0] + " <- " + migliore[1],\n            p_b=round(p_b, 4), f_b=round(f_b, 4)))\n        dettaglio_nullo[dis + "_" + str(seed)] = gd.tolist()\n        print(("[" + dis + " mondo " + str(seed) + "]").ljust(22) +\n              "g_b " + format(gb, "+9.1f") + " (" + format(100 * gb / Ya, "+.3f") + "%)" +\n              " | nullo mediana " + format(float(np.median(gd)), "+8.1f") +\n              " p90 " + format(float(np.quantile(gd, 0.90)), "+8.1f") +\n              " | oracolo " + format(ge, "+8.1f") +\n              " | p_b " + format(p_b, ".3f") + " f_b " + format(f_b, ".3f"))\n    with open(os.path.join(QUI, "D11_risultati.csv"), "w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(righe_out[0])); w.writeheader(); w.writerows(righe_out)\n    json.dump(dettaglio_nullo, open(os.path.join(QUI, "D11_controllo_nullo.json"), "w"))\n    # --- i conteggi pre-registrati\n    def conta(ins):\n        V = sum(1 for r in ins if r["g_b"] > 0)\n        S = sum(1 for r in ins if r["p_b"] >= 0.90)\n        W = sum(1 for r in ins if r["g_b_per_euro"] > r["g_c_per_euro"])\n        return V, S, W\n    tutti = righe_out\n    non_visti = [r for r in righe_out if not r["gia_visto"]]\n    print()\n    for et, ins in (("tutti e 8 i casi", tutti), ("i 7 non visti", non_visti)):\n        V, S, W = conta(ins)\n        N = len(ins)\n        soglia = 6 if N == 8 else int(6.0 / 8.0 * N + 0.5)\n        print(et.ljust(20) + "N=" + str(N) + "  V=" + str(V) + "  S=" + str(S) + "  W=" + str(W) +\n              "  (soglia " + str(soglia) + ")" +\n              "  mediane: g_b " + format(np.median([r["g_b_pct"] for r in ins]), "+.3f") + "%" +\n              "  p_b " + format(np.median([r["p_b"] for r in ins]), ".3f") +\n              "  f_b " + format(np.median([r["f_b"] for r in ins]), ".3f"))\n    V, S, W = conta(tutti)\n    ramo = ("1 LA REGOLA GUADAGNA E BATTE IL CASO" if S >= 6 else\n            "2 NON CONCLUSIVO" if 3 <= S <= 5 else\n            "3 DIREZIONE GIUSTA, GUADAGNO INDISTINGUIBILE DAL CASO" if V >= 6 else\n            "4 LA REGOLA NON GUADAGNA")\n    print()\n    print("RAMO CALCOLATO SULLA REGOLA PRE-REGISTRATA: " + ramo)\n\n\nif __name__ == "__main__":\n    main()\n'
S11B = '"""D11b - regole di riparto alternative, sullo stesso banco di D11.\n\nRiusa studio_D11.py (che NON viene modificato): stessi 8 casi, stessa verita\'\n(superficie di risposta del mondo base, beta FISSO), stesso controllo nullo\n(200 coppie casuali della stessa cifra), stesso oracolo (miglior coppia).\nAggiunge quattro regole:\n\n  B1  aggregato a 4 voci (Google / Meta / LinkedIn / Job board), poi 90/10\n  B2  solo tagli: taglia dove P<=10%, ridistribuisce in proporzione alla spesa\n  H9  passo scalato con la certezza: passo_c = 15% x (1 - w_c / w_max)\n  H6  il totale: tutti i canali x k, sulla verita\'; ROI del totale dai draw\n\nuso: python studio_D11b.py --cancello   # riproduce (a) e (b) di D11, e basta\n     python studio_D11b.py              # lo studio intero (DOPO l\'approvazione)\n"""\nimport csv\nimport gzip\nimport importlib.util\nimport json\nimport os\nimport sys\n\nimport numpy as np\n\nQUI = os.path.dirname(os.path.abspath(__file__))\n_s = importlib.util.spec_from_file_location("s11", os.path.join(QUI, "studio_D11.py"))\nS11 = importlib.util.module_from_spec(_s); _s.loader.exec_module(S11)\nG, PB, PK, CASI = S11.G, S11.PB, S11.PK, S11.CASI\n\nAGGREGATI = {"Google": ["Google Ads"], "Meta": ["Meta Ads"], "LinkedIn": ["LinkedIn Ads"],\n             "Job board": ["Indeed", "Subito Lavoro", "Jooble", "Altre job board"]}\nPASSO = PB.PASSO_PCT                       # 15%, lo stesso della 90/10\nK_TOTALE = [0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20]\nRPK = 40.0                                 # revenue_per_kpi del runner\nREGOLE = ["b", "B1", "B2", "H9"]\nTOLL_SOMMA_EUR = 1.0\n\n\n# ----------------------------------------------------------------- i draw\ndef draw_matrice(draw_path):\n    """(n_draw x canali) contributi, piu\' la colonna totale, dai draw salvati."""\n    with gzip.open(draw_path, "rt", newline="", encoding="utf-8") as f:\n        r = csv.DictReader(f)\n        canali = [c for c in r.fieldnames if c != "totale"]\n        righe = [[float(x[c]) for c in canali] + [float(x["totale"])] for x in r]\n    M = np.array(righe)\n    return canali, M[:, :-1], M[:, -1]\n\n\ndef spesa_del_fit(can, seed):\n    return {x["canale"]: float(x["spesa"]) for x in\n            csv.DictReader(open(os.path.join(PK, can), newline="", encoding="utf-8"))\n            if x["seed_mondo"] == str(seed)}\n\n\n# ----------------------------------------------------------------- le regole\ndef riparto_B1(draw_path, spesa):\n    """90/10 su quattro aggregati; dentro l\'aggregato la stessa percentuale a tutti."""\n    canali, C, T = draw_matrice(draw_path)\n    idx = {c: i for i, c in enumerate(canali)}\n    sp_agg = {a: sum(spesa[c] for c in cs) for a, cs in AGGREGATI.items()}\n    roi_port = T / sum(spesa.values())\n    P = {}\n    for a, cs in AGGREGATI.items():\n        contrib = C[:, [idx[c] for c in cs]].sum(axis=1)\n        P[a] = float((contrib / sp_agg[a] > roi_port).mean() * 100.0)\n    righe, scambio = PB.proposta(P, sp_agg)          # stessa regola di bilancio della 90/10\n    rip = {}\n    for a, p, dec, d, nu, no in righe:\n        if dec in ("ALZA", "TAGLIA"):\n            for c in AGGREGATI[a]:\n                rip[c] = d\n    return rip, P\n\n\ndef riparto_B2(draw_path, spesa):\n    """Solo il lato del taglio della 90/10; i soldi liberati vanno a tutti gli altri\n    in proporzione alla spesa attuale (= stessa percentuale per tutti i non tagliati)."""\n    P, n = PB.probabilita(draw_path, spesa)\n    tagli = [c for c, p in P.items() if p <= PB.SOGLIA_TAGLIA]\n    if not tagli:\n        return {}, P\n    liberati = sum(spesa[c] * PASSO / 100.0 for c in tagli)\n    altri = [c for c in spesa if c not in tagli]\n    sp_altri = sum(spesa[c] for c in altri)\n    rip = {c: -PASSO for c in tagli}\n    for c in altri:\n        rip[c] = 100.0 * liberati / sp_altri\n    return rip, P\n\n\ndef riparto_H9(draw_path, spesa):\n    """La 90/10 con passo per canale scalato dalla certezza: 1 - w_c / w_max, dove\n    w_c = (q95 - q05) / mediana del ROI del canale nei draw. Il canale con\n    l\'intervallo piu\' largo ha passo zero. Bilancio: si scala il lato piu\' grande."""\n    canali, C, T = draw_matrice(draw_path)\n    idx = {c: i for i, c in enumerate(canali)}\n    P, n = PB.probabilita(draw_path, spesa)\n    w = {}\n    for c in canali:\n        roi = C[:, idx[c]] / spesa[c]\n        w[c] = float((np.quantile(roi, .95) - np.quantile(roi, .05)) / np.median(roi))\n    w_max = max(w.values())\n    passo = {c: PASSO * (1.0 - w[c] / w_max) for c in canali}\n    desiderio = {c: ("ALZA" if p >= PB.SOGLIA_ALZA else "TAGLIA")\n                 for c, p in P.items() if p >= PB.SOGLIA_ALZA or p <= PB.SOGLIA_TAGLIA}\n    su = sum(spesa[c] * passo[c] / 100.0 for c, v in desiderio.items() if v == "ALZA")\n    giu = sum(spesa[c] * passo[c] / 100.0 for c, v in desiderio.items() if v == "TAGLIA")\n    scambio = min(su, giu)\n    k_su = scambio / su if su > 0 else 0.0\n    k_giu = scambio / giu if giu > 0 else 0.0\n    rip = {}\n    for c, v in desiderio.items():\n        eff = passo[c] * (k_su if v == "ALZA" else -k_giu)\n        if abs(eff) > 1e-12:\n            rip[c] = eff\n    return rip, P, w, passo\n\n\ndef totale_H6(curve, canali, spesa, draw_path):\n    """Tutti i canali x k, sulla verita\'. Piu\' il ROI del totale nei draw."""\n    Y = {k: S11.contributo(curve, canali, {c: 100.0 * (k - 1.0) for c in canali}) for k in K_TOTALE}\n    tot_sp = sum(spesa.values())\n    marg_vero = (Y[1.05] - Y[0.95]) / (0.10 * tot_sp) * RPK      # EUR per EUR, al totale attuale\n    roi_medio_vero = Y[1.00] * RPK / tot_sp\n    _, C, T = draw_matrice(draw_path)\n    roi_tot = T / tot_sp\n    return Y, marg_vero, roi_medio_vero, (float(np.quantile(roi_tot, .05)),\n                                          float(np.median(roi_tot)),\n                                          float(np.quantile(roi_tot, .95)))\n\n\n# ----------------------------------------------------------------- il banco\ndef valuta(curve, canali, spesa, rip, Ya, seed):\n    """g, euro mossi, nullo e oracolo alla STESSA cifra della regola, percentile, frazione.\n    Un riparto vuoto (\'fermo\') ha g = 0 e nessun nullo: p e f restano NaN."""\n    eur = sum(abs(spesa[c] * d / 100.0) for c, d in rip.items()) / 2.0\n    Y = S11.contributo(curve, canali, rip)\n    g = Y - Ya\n    if eur <= 0:\n        return dict(Y=Y, g=g, eur=0.0, gd=np.array([]), ge=float("nan"), oracolo="",\n                    p=float("nan"), f=float("nan"))\n    rng = np.random.default_rng(S11.SEME_NULLO + seed)       # stesso seme di D11\n    tutte = S11.coppie(canali)\n    gd = []\n    while len(gd) < S11.N_CASUALI:\n        su, giu = tutte[int(rng.integers(0, len(tutte)))]\n        d_giu = -100.0 * eur / spesa[giu]\n        if d_giu <= -100.0:\n            continue\n        gd.append(S11.contributo(curve, canali, {su: 100.0 * eur / spesa[su], giu: d_giu}) - Ya)\n    gd = np.array(sorted(gd))\n    migliore, ge = None, -np.inf\n    for su, giu in tutte:\n        d_giu = -100.0 * eur / spesa[giu]\n        if d_giu <= -100.0:\n            continue\n        gg = S11.contributo(curve, canali, {su: 100.0 * eur / spesa[su], giu: d_giu}) - Ya\n        if gg > ge:\n            ge, migliore = gg, (su, giu)\n    return dict(Y=Y, g=g, eur=eur, gd=gd, ge=ge, oracolo=migliore[0] + " <- " + migliore[1],\n                p=float((gd < g).mean()), f=(g / ge if ge > 0 else float("nan")))\n\n\ndef controlla_somma(rip, spesa, nome):\n    scarto = sum(spesa[c] * d / 100.0 for c, d in rip.items())\n    if abs(scarto) > TOLL_SOMMA_EUR:\n        raise SystemExit(nome + ": la somma del riparto non e\' zero: " + format(scarto, "+.2f") + " EUR")\n    return scarto\n\n\ndef testo_rip(rip):\n    return ";".join(c + "=" + format(d, "+.4f") for c, d in rip.items())\n\n\ndef cancello():\n    """Riproduce (a) e (b) di D11 e le confronta con D11_risultati.csv. Numeri gia\' noti."""\n    attesi = {(r["disegno"], int(r["mondo"])): r for r in\n              csv.DictReader(open(os.path.join(QUI, "D11_risultati.csv"), newline="", encoding="utf-8"))}\n    ok = True\n    print("CANCELLO D11b: riproduzione di (a) spesa attuale e (b) regola 90/10 di D11")\n    for dis, seed, draw, can in CASI:\n        canali, spesa, curve = S11.mondo(seed)\n        rip_b, _, _ = S11.riparto_90_10(draw, can, seed)\n        Ya = S11.contributo(curve, canali)\n        vb = valuta(curve, canali, spesa, rip_b, Ya, seed)\n        a = attesi[(dis, seed)]\n        d = [abs(Ya - float(a["Y_a"])), abs(vb["Y"] - float(a["Y_b"])),\n             abs(vb["g"] - float(a["g_b"])), abs(vb["p"] - float(a["p_b"])),\n             abs(vb["eur"] - float(a["euro_mossi_b"]))]\n        bene = d[0] < 1e-2 and d[1] < 1e-2 and d[2] < 1e-2 and d[3] < 1e-3 and d[4] < 1e-1\n        ok &= bene\n        print(("  [" + dis + " " + str(seed) + "]").ljust(14) +\n              ("OK " if bene else "DIVERSO ") +\n              "Y_a " + format(Ya, ".3f") + " Y_b " + format(vb["Y"], ".3f") +\n              " g_b " + format(vb["g"], "+.3f") + " p_b " + format(vb["p"], ".4f") +\n              " eur " + format(vb["eur"], ".2f"))\n    print("CANCELLO: " + ("PASSATO, 8 su 8 identici a D11" if ok else "FALLITO"))\n    return ok\n\n\ndef main():\n    if not cancello():\n        raise SystemExit("il cancello non passa: non si va avanti")\n    print()\n    righe_out, nulli, h6 = [], {}, []\n    for dis, seed, draw, can in CASI:\n        canali, spesa, curve = S11.mondo(seed)\n        dp = os.path.join(PK, draw)\n        Ya = S11.contributo(curve, canali)\n        rip = {}\n        rip["b"], _, _ = S11.riparto_90_10(draw, can, seed)\n        rip["B1"], P1 = riparto_B1(dp, spesa_del_fit(can, seed))\n        rip["B2"], P2 = riparto_B2(dp, spesa_del_fit(can, seed))\n        rip["H9"], P9, w9, passo9 = riparto_H9(dp, spesa_del_fit(can, seed))\n        riga = dict(disegno=dis, mondo=seed, gia_visto=int((dis, seed) == S11.CASO_GIA_VISTO),\n                    Y_a=round(Ya, 3))\n        for nome in REGOLE:\n            scarto = controlla_somma(rip[nome], spesa, nome + " " + dis + "/" + str(seed))\n            v = valuta(curve, canali, spesa, rip[nome], Ya, seed)\n            riga[nome + "_riparto"] = testo_rip(rip[nome])\n            riga[nome + "_scarto_eur"] = round(scarto, 4)\n            riga[nome + "_eur"] = round(v["eur"], 2)\n            riga[nome + "_Y"] = round(v["Y"], 3)\n            riga[nome + "_g"] = round(v["g"], 3)\n            riga[nome + "_g_pct"] = round(100.0 * v["g"] / Ya, 5)\n            riga[nome + "_p"] = round(v["p"], 4) if v["p"] == v["p"] else ""\n            riga[nome + "_f"] = round(v["f"], 4) if v["f"] == v["f"] else ""\n            riga[nome + "_oracolo"] = v["oracolo"]\n            riga[nome + "_nullo_mediana"] = round(float(np.median(v["gd"])), 3) if len(v["gd"]) else ""\n            riga[nome + "_nullo_p90"] = round(float(np.quantile(v["gd"], .90)), 3) if len(v["gd"]) else ""\n            nulli[nome + "_" + dis + "_" + str(seed)] = v["gd"].tolist()\n        riga["H9_larghezze"] = ";".join(c + "=" + format(w9[c], ".3f") for c in canali)\n        riga["H9_passi"] = ";".join(c + "=" + format(passo9[c], ".2f") for c in canali)\n        riga["B1_P"] = ";".join(a + "=" + format(P1[a], ".1f") for a in AGGREGATI)\n        righe_out.append(riga)\n        Y6, marg, roi_med, (q05, q50, q95) = totale_H6(curve, canali, spesa, dp)\n        h6.append(dict(disegno=dis, mondo=seed, spesa_totale=round(sum(spesa.values()), 2),\n                       roi_medio_vero=round(roi_med, 4), marginale_totale_vero=round(marg, 4),\n                       roi_totale_draw_q05=round(q05, 4), roi_totale_draw_q50=round(q50, 4),\n                       roi_totale_draw_q95=round(q95, 4),\n                       **{"Y_k" + format(k, ".2f"): round(Y6[k], 3) for k in K_TOTALE},\n                       **{"g_pct_k" + format(k, ".2f"): round(100.0 * (Y6[k] / Y6[1.0] - 1.0), 4) for k in K_TOTALE}))\n        print(("[" + dis + " " + str(seed) + "]").ljust(12) +\n              " ".join(n + " " + format(riga[n + "_g_pct"], "+.3f") + "%" +\n                       ("(p " + format(riga[n + "_p"], ".2f") + ")" if riga[n + "_p"] != "" else "(fermo)")\n                       for n in REGOLE) +\n              " | H6 marg " + format(marg, ".3f") + " vs medio " + format(roi_med, ".3f"))\n    with open(os.path.join(QUI, "D11b_risultati.csv"), "w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(righe_out[0])); w.writeheader(); w.writerows(righe_out)\n    with open(os.path.join(QUI, "D11b_totale_H6.csv"), "w", newline="", encoding="utf-8") as f:\n        w = csv.DictWriter(f, fieldnames=list(h6[0])); w.writeheader(); w.writerows(h6)\n    json.dump(nulli, open(os.path.join(QUI, "D11b_controllo_nullo.json"), "w"))\n    # --- i conteggi e i rami pre-registrati, per regola\n    print()\n    print("CONTEGGI (8 casi). S = casi con p >= 0,90. W\' = casi con g% > g% della 90/10. F = casi fermi.")\n    for nome in ("B1", "B2", "H9"):\n        S = sum(1 for r in righe_out if r[nome + "_p"] != "" and r[nome + "_p"] >= 0.90)\n        W = sum(1 for r in righe_out if r[nome + "_g_pct"] > r["b_g_pct"])\n        F = sum(1 for r in righe_out if r[nome + "_eur"] == 0)\n        V = sum(1 for r in righe_out if r[nome + "_g"] > 0)\n        ramo = ("1 GUADAGNA E BATTE IL CASO" if S >= 6 else\n                "2 MEGLIO DELLA 90/10, NON DEL CASO" if W >= 6 else\n                "3 NON MEGLIO DELLA 90/10")\n        peggiore = min(r[nome + "_g_pct"] for r in righe_out)\n        peggiore_b = min(r["b_g_pct"] for r in righe_out)\n        print("  " + nome.ljust(3) + " V=" + str(V) + " S=" + str(S) + " W\'=" + str(W) + " F=" + str(F) +\n              "  mediana g " + format(float(np.median([r[nome + "_g_pct"] for r in righe_out])), "+.3f") + "%" +\n              "  perdita max " + format(peggiore, "+.3f") + "% (90/10: " + format(peggiore_b, "+.3f") + "%)" +\n              "  -> RAMO " + ramo)\n    m = [r["marginale_totale_vero"] for r in h6]\n    print("  H6  marginale vero sul totale: mediana " + format(float(np.median(m)), ".3f") +\n          " EUR/EUR (min " + format(min(m), ".3f") + " max " + format(max(m), ".3f") + ")" +\n          "; ROI medio vero " + format(float(np.median([r["roi_medio_vero"] for r in h6])), ".3f"))\n\n\nif __name__ == "__main__":\n    if "--cancello" in sys.argv:\n        sys.exit(0 if cancello() else 1)\n    main()\n'
io.open('studio_D11.py', 'w', encoding='utf-8').write(S11)
io.open('studio_D11b.py', 'w', encoding='utf-8').write(S11B)
print('studio_D11.py  sha256[:16] =', hashlib.sha256(S11.encode()).hexdigest()[:16])
print('studio_D11b.py sha256[:16] =', hashlib.sha256(S11B.encode()).hexdigest()[:16])
# studio_D11.py cerca il pacchetto in QUI/D11_pacchetto e il generatore in QUI/g5: gia' cosi'.


In [ ]:
# ============================================================================
# CELLA 3 - IL CANCELLO: riproduce (a) e (b) di D11. Numeri gia' noti, nessuna regola nuova.
# ============================================================================
import subprocess, sys
r = subprocess.run([sys.executable, 'studio_D11b.py', '--cancello'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)
if r.returncode != 0:
    raise SystemExit("IL CANCELLO NON PASSA: fermarsi qui e capire perche'.")
print("Cancello passato. Da qui in poi serve l'approvazione della pre-registrazione.")


In [ ]:
# ============================================================================
# CELLA 4 - LO STUDIO. Bloccata finche' la pre-registrazione non e' approvata.
# ============================================================================
APPROVATO = False      # <-- metti True SOLO dopo l'approvazione scritta di Giacomo
if not APPROVATO:
    raise SystemExit('APPROVATO = False: la pre-registrazione va approvata prima di calcolare numeri nuovi.')
import subprocess, sys
r = subprocess.run([sys.executable, 'studio_D11b.py'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)
if r.returncode != 0:
    raise SystemExit('studio fallito')


In [ ]:
# ============================================================================
# CELLA 5 - LETTURA: la tabella per caso e i conteggi, cosi' come escono
# ============================================================================
import csv
righe = list(csv.DictReader(open('D11b_risultati.csv', newline='', encoding='utf-8')))
print('%-9s %-6s %8s %8s %8s %8s   %6s %6s %6s' % ('disegno','mondo','b','B1','B2','H9','p_B1','p_B2','p_H9'))
for r in righe:
    print('%-9s %-6s %+8.3f %+8.3f %+8.3f %+8.3f   %6s %6s %6s' % (
        r['disegno'], r['mondo'], float(r['b_g_pct']), float(r['B1_g_pct']), float(r['B2_g_pct']), float(r['H9_g_pct']),
        r['B1_p'] or 'fermo', r['B2_p'] or 'fermo', r['H9_p'] or 'fermo'))
print()
for r in csv.DictReader(open('D11b_totale_H6.csv', newline='', encoding='utf-8')):
    print('H6 %-4s %-4s marginale vero sul totale %s EUR/EUR | ROI medio vero %s | ROI totale nei draw q50 %s' % (
        r['disegno'], r['mondo'], r['marginale_totale_vero'], r['roi_medio_vero'], r['roi_totale_draw_q50']))
print()
print('I rami sono stampati dalla cella 4. Si leggono SOLO contro la pre-registrazione.')
